# CLAUD·IA — Premier fine-tuning supervisé

**Parole espagnole → texte anglais · SeamlessM4T v2**

Ce notebook pilote entraîne deux projections du dernier bloc d’attention du décodeur. Les autres poids restent gelés. Les 16 paires d’entraînement servent à faire des mises à jour réelles. Les 4 paires de validation servent à comparer la perte avant et après. Ce petit lot permet de vérifier la méthode et la mémoire disponible.

[Documentation de SeamlessM4T v2](https://huggingface.co/docs/transformers/model_doc/seamless_m4t_v2) · [Données FLEURS](https://huggingface.co/datasets/google/fleurs)

## 1. Préparer Colab

Choisir **Runtime → Change runtime type → T4 GPU**. Exécuter l’installation, puis redémarrer le noyau si Colab le demande. Importer `fleurs_pilote.zip` via le panneau **Fichiers** une fois l’environnement GPU actif.

In [ ]:
%pip install "torch>=2.6" "transformers[sentencepiece]==5.22.1" "huggingface-hub>=1.32,<2" soundfile

## 2. Vérifier la GPU et ouvrir le lot pilote

Le notebook lit l’archive produite par le notebook 02. Il vérifie les fréquences et la séparation des phrases entre entraînement et validation.

In [ ]:
import json
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

import soundfile as sf
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Sélectionnez une GPU T4 dans Colab, puis relancez cette cellule.")

device = torch.device("cuda")
archive_path = Path("fleurs_pilote.zip")
sample_rate = 16_000
if not archive_path.is_file():
    raise FileNotFoundError("Importez fleurs_pilote.zip depuis le panneau Fichiers.")

def read_split(archive, split):
    examples = []
    for line in archive.read(f"{split}.jsonl").decode("utf-8").splitlines():
        record = json.loads(line)
        audio, rate = sf.read(BytesIO(archive.read(record["audio_path"])), dtype="float32")
        if rate != sample_rate or audio.ndim != 1 or not 0 < len(audio) <= 8 * rate:
            raise ValueError(f"Audio inattendu pour la phrase {record['sentence_id']}.")
        examples.append({**record, "audio": audio})
    return examples

with ZipFile(archive_path) as archive:
    train_examples = read_split(archive, "train")
    validation_examples = read_split(archive, "validation")

train_ids = {example["sentence_id"] for example in train_examples}
validation_ids = {example["sentence_id"] for example in validation_examples}
if len(train_examples) != 16 or len(validation_examples) != 4:
    raise ValueError("Le lot pilote doit contenir 16 paires train et 4 paires validation.")
if len(train_ids) != 16 or len(validation_ids) != 4 or train_ids & validation_ids:
    raise ValueError("Des identifiants de phrase sont répétés.")

print(torch.cuda.get_device_name(0))
print(f"{len(train_examples)} paires train, {len(validation_examples)} validation.")

## 3. Charger le modèle et sélectionner les poids à entraîner

Nous chargeons la version **parole → texte**. Dans la dernière couche du décodeur, nous entraînons les projections **requête** et **valeur** de l’attention vers l’audio. La requête aide le décodeur à repérer l’information utile dans la représentation audio. La valeur transporte l’information retenue.

Nous gardons les poids gelés en FP16 et les poids entraînés en FP32. Si le cache est vide, le chargement télécharge les fichiers du modèle.

In [ ]:
from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToText

model_id = "facebook/seamless-m4t-v2-large"
target_language = "eng"

processor = AutoProcessor.from_pretrained(model_id)
model = SeamlessM4Tv2ForSpeechToText.from_pretrained(
    model_id, dtype=torch.float16, use_safetensors=True
).to(device)
model.requires_grad_(False)
model.eval()

last_layer = model.text_decoder.layers[-1]
for projection in (last_layer.cross_attention.q_proj, last_layer.cross_attention.v_proj):
    projection.float().requires_grad_(True)

trainable = {name: parameter for name, parameter in model.named_parameters() if parameter.requires_grad}
initial_weights = {name: parameter.detach().cpu().clone() for name, parameter in trainable.items()}
print(f"Paramètres entraînables : {sum(parameter.numel() for parameter in trainable.values()):,}")
print("\n".join(trainable))

## 4. Mesurer avant le fine-tuning

Pour calculer la perte, le décodeur reçoit le début correct de la phrase et doit prédire chaque unité suivante. La cible anglaise sert à calculer la perte. Pour la traduction affichée, le modèle reçoit uniquement l’audio et le code de langue. Nous gardons les mêmes conditions avant et après.

In [ ]:
from statistics import mean

language_id = model.generation_config.text_decoder_lang_to_code_id[target_language]

def audio_inputs(example):
    inputs = processor(
        audio=example["audio"], sampling_rate=sample_rate, return_tensors="pt"
    ).to(device)
    inputs["input_features"] = inputs["input_features"].to(dtype=torch.float16)
    return inputs

def supervised_loss(example):
    inputs = audio_inputs(example)
    token_ids = processor.tokenizer(example["target_text"], add_special_tokens=False).input_ids
    decoder_ids = torch.tensor(
        [[model.config.decoder_start_token_id, language_id, *token_ids]], device=device
    )
    labels = torch.tensor([[-100, *token_ids, model.config.eos_token_id]], device=device)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        return model(**inputs, decoder_input_ids=decoder_ids, labels=labels, use_cache=False).loss

@torch.no_grad()
def validation_loss():
    model.eval()
    return mean(supervised_loss(example).item() for example in validation_examples)

@torch.no_grad()
def translate(example):
    model.eval()
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        tokens = model.generate(
            **audio_inputs(example), tgt_lang=target_language, max_new_tokens=80,
            return_dict_in_generate=False,
        )
    return processor.batch_decode(
        tokens.cpu(), skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

validation_before = validation_loss()
example = validation_examples[0]
prediction_before = translate(example)
print(f"Perte moyenne de validation avant : {validation_before:.4f}")
print(f"Référence : {example['target_text']}")
print(f"Prédiction avant : {prediction_before}")

## 5. Entraîner et comparer

Une époque parcourt les 16 paires une fois, avec un lot d’une phrase. La rétropropagation calcule les gradients et AdamW ajuste les poids sélectionnés. La mise à l’échelle des gradients facilite l’entraînement en précision mixte. Le mode évaluation désactive les tirages aléatoires des couches pendant ce pilote, tout en laissant la rétropropagation active.

Cette cellule repart des poids préentraînés si vous la relancez. La variation de la perte sur quatre phrases reste indicative.

In [ ]:
import math
import random

learning_rate = 3e-5
rng = random.Random(42)
order = train_examples.copy()
rng.shuffle(order)

with torch.no_grad():
    for name, parameter in trainable.items():
        parameter.copy_(initial_weights[name])

optimizer = torch.optim.AdamW(trainable.values(), lr=learning_rate, weight_decay=0)
scaler = torch.amp.GradScaler("cuda", init_scale=1024)
train_losses = []

for step, example in enumerate(order, start=1):
    optimizer.zero_grad(set_to_none=True)
    loss = supervised_loss(example)
    loss_value = loss.item()
    if not math.isfinite(loss_value):
        raise RuntimeError(f"Perte non finie à l’étape {step}.")
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    train_losses.append(loss_value)
    print(f"Étape {step:02d}/{len(order)} — perte : {loss_value:.4f}")

updated_names = [
    name for name, parameter in trainable.items()
    if not torch.equal(parameter.detach().cpu(), initial_weights[name])
]
if not updated_names:
    raise RuntimeError("Aucun poids entraînable n’a changé pendant le pilote.")

validation_after = validation_loss()
example = validation_examples[0]
prediction_after = translate(example)
print(f"Paramètres modifiés : {len(updated_names)} tenseurs")
print(f"Perte de validation avant : {validation_before:.4f}")
print(f"Perte de validation après : {validation_after:.4f}")
print(f"Référence : {example['target_text']}")
print(f"Prédiction avant : {prediction_before}")
print(f"Prédiction après : {prediction_after}")

## 6. Sauvegarder le résultat

L’archive contient uniquement les poids ajustés et le rapport d’essai. Elle pourra servir à reproduire la comparaison et à réintégrer les poids dans le modèle vocal. Téléchargez-la avant de fermer la session Colab.

In [ ]:
import shutil
from safetensors.torch import save_file

output_dir = Path("fine_tuning_pilote")
output_dir.mkdir(exist_ok=True)
save_file(
    {name: parameter.detach().cpu().contiguous() for name, parameter in trainable.items()},
    output_dir / "poids_ajustes.safetensors",
)
report = {
    "base_model": model_id,
    "gpu": torch.cuda.get_device_name(0),
    "source_language": "spa",
    "target_language": target_language,
    "learning_rate": learning_rate,
    "train_sentence_ids": sorted(train_ids),
    "validation_sentence_ids": sorted(validation_ids),
    "training_losses": train_losses,
    "validation_loss_before": validation_before,
    "validation_loss_after": validation_after,
    "validation_example_id": example["sentence_id"],
    "reference": example["target_text"],
    "prediction_before": prediction_before,
    "prediction_after": prediction_after,
    "updated_parameters": updated_names,
}
(output_dir / "rapport.json").write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
archive = shutil.make_archive(output_dir.name, "zip", root_dir=output_dir)
print(f"Archive prête : {Path(archive).name}")